In [1]:
import duckdb
import polars as pl
import polars.selectors as cs

In [2]:
# get predictions from python and database
with duckdb.connect('../dev.duckdb') as con:
    df_pred_db = con.table("pred_churn").pl()

df_pred_py = pl.read_csv('preds_py.csv')

df_preds = df_pred_db.join(df_pred_py, on = 'customer_id', suffix = '_py')
df_preds.head()

pred,customer_id,pred_py
"decimal[29,19]",str,f64
0.4535526260733604350,"""7590-VHVEG""",0.453553
0.1158851841464638720,"""5575-GNVDE""",0.115885
0.3124912194907665290,"""3668-QPYBK""",0.312491
0.1047914009541273120,"""7795-CFOCW""",0.104791
0.5674070119857788010,"""9237-HQITU""",0.567407


In [3]:
# find any differences
(
df_preds
.filter( (pl.col('pred') - pl.col('pred_py')).abs() > 0.0001)
.select('customer_id', 'pred_py', pl.col('pred').alias('pred_db'))
.with_columns( cs.numeric().cast(pl.Float32).round(4) )
)

customer_id,pred_py,pred_db
str,f32,f32
"""4472-LVYGI""",0.1317,0.116
"""5709-LVOEQ""",0.1762,0.1605
"""1371-DWPAZ""",0.1277,0.112
"""4075-WKNIU""",0.2973,0.2816
"""2775-SEFEE""",0.1853,0.1696
